# Airline Passenger Satisfaction — End-to-End ML Pipeline

This notebook covers the complete workflow:
1. Data Loading & Inspection
2. Missing Value Imputation & Survey Data Cleaning
3. Feature Engineering & Transformations
4. Categorical Encoding & Scaling
5. Stratified Train / Validation / Test Splitting
6. Model Training & Comparison (LightGBM, XGBoost, Random Forest)
7. Model Evaluation, Feature Importance & Single-Point Inference

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

print('Libraries imported successfully.')

## 1. Load Dataset & Inspection

In [ ]:
df = pd.read_csv('airline_passenger_satisfaction.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
# Check null values
print('Missing Values:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nTarget Distribution:\n', df['Satisfaction'].value_counts(normalize=True))

## 2. Data Cleaning
- Drop `ID` index
- Median impute `Arrival Delay`
- Replace `0` in Likert rating features with column median (0 = Not Applicable)

In [ ]:
# Drop unneeded ID column
df.drop(columns=['ID'], inplace=True)

# Impute missing arrival delays with median
df['Arrival Delay'] = df['Arrival Delay'].fillna(df['Arrival Delay'].median())

# Handle 0s in survey rating features
rating_cols = [
    'Departure and Arrival Time Convenience', 'Ease of Online Booking', 'Check-in Service',
    'Online Boarding', 'Gate Location', 'On-board Service', 'Seat Comfort', 'Leg Room Service',
    'Cleanliness', 'Food and Drink', 'In-flight Service', 'In-flight Wifi Service',
    'In-flight Entertainment', 'Baggage Handling'
]

for col in rating_cols:
    median_val = df[col].replace(0, np.nan).median()
    df[col] = df[col].replace(0, np.nan).fillna(median_val)

print('Total nulls remaining:', df.isnull().sum().sum())

## 3. Feature Engineering
- Combine Departure + Arrival Delay into `Total Delay`
- Log transform `Flight Distance` (normalizes positive skew)
- Age Bins discretization
- Domain composite scores: `Ground Service Score` and `In-flight Service Score`

In [ ]:
# Total Delay
df['Total Delay'] = df['Departure Delay'] + df['Arrival Delay']
df.drop(columns=['Departure Delay', 'Arrival Delay'], inplace=True)

# Log Flight Distance
df['Log Flight Distance'] = np.log1p(df['Flight Distance'])
df.drop(columns=['Flight Distance'], inplace=True)

# Age Group Bins
df['Age Group'] = pd.cut(
    df['Age'],
    bins=[0, 17, 25, 40, 60, 100],
    labels=['Child', 'Young Adult', 'Adult', 'Middle-Aged', 'Senior']
)

# Composite Scores
ground_cols = ['Departure and Arrival Time Convenience', 'Ease of Online Booking', 'Check-in Service', 'Gate Location']
inflight_cols = ['Online Boarding', 'On-board Service', 'Seat Comfort', 'Leg Room Service', 'Cleanliness', 'Food and Drink', 'In-flight Service', 'In-flight Wifi Service', 'In-flight Entertainment', 'Baggage Handling']

df['Ground Service Score'] = df[ground_cols].mean(axis=1).round(2)
df['In-flight Service Score'] = df[inflight_cols].mean(axis=1).round(2)

print('Dataset shape after feature engineering:', df.shape)

## 4. Encoding Categoricals & Feature Scaling

In [ ]:
# Binary Encoding
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['Customer Type'] = df['Customer Type'].map({'First-time': 0, 'Returning': 1})
df['Type of Travel'] = df['Type of Travel'].map({'Personal': 0, 'Business': 1})
df['Satisfaction'] = df['Satisfaction'].map({'Neutral or Dissatisfied': 0, 'Satisfied': 1})

# One-Hot Encoding
df = pd.get_dummies(df, columns=['Class'], prefix='Class', drop_first=False)
df = pd.get_dummies(df, columns=['Age Group'], prefix='AgeGrp', drop_first=False)
df.drop(columns=['Age'], inplace=True)

# Scaling
scale_cols = ['Total Delay', 'Log Flight Distance', 'Ground Service Score', 'In-flight Service Score'] + rating_cols
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print('Final Processed Features Count:', df.shape[1] - 1)

## 5. Stratified Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
X = df.drop(columns=['Satisfaction'])
y = df['Satisfaction']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print('Train Set :', X_train.shape)
print('Val Set   :', X_val.shape)
print('Test Set  :', X_test.shape)

## 6. Model Training & Comparison

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63, max_depth=-1,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=10, reg_alpha=0.5,
        class_weight='balanced', random_state=42, verbose=-1, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=8, subsample=0.9,
        colsample_bytree=0.7, min_child_weight=5, gamma=0.1, reg_lambda=1.0,
        scale_pos_weight=pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, max_depth=20, min_samples_split=5, min_samples_leaf=2,
        max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1
    )
}

results = []
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    for split_name, xs, ys in [('Train', X_train, y_train), ('Validation', X_val, y_val), ('Test', X_test, y_test)]:
        preds = model.predict(xs)
        probs = model.predict_proba(xs)[:, 1]
        results.append({
            'Model': name,
            'Split': split_name,
            'Accuracy': round(accuracy_score(ys, preds), 4),
            'F1_Score': round(f1_score(ys, preds), 4),
            'ROC_AUC': round(roc_auc_score(ys, probs), 4)
        })

df_results = pd.DataFrame(results)
df_results

## 7. Test Set Evaluation & Feature Importance

In [ ]:
best_model = trained_models['LightGBM']
y_test_pred = best_model.predict(X_test)

print('=== LightGBM Classification Report on Test Set ===')
print(classification_report(y_test, y_test_pred, target_names=['Neutral/Dissatisfied', 'Satisfied']))

# Top 15 Feature Importances
fi = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
fi.plot(kind='bar', color='#00f2fe', edgecolor='#0b1122')
plt.title('Top 15 Most Important Features (LightGBM)', fontsize=12, fontweight='bold')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Test Point Prediction (Single Passenger Inference)

In [ ]:
# Test on a sample passenger
sample = X_test.iloc[[1]]
actual = 'Satisfied' if y_test.iloc[1] == 1 else 'Dissatisfied'

print(f'Actual Passenger Experience: {actual}')
for name, model in trained_models.items():
    pred = model.predict(sample)[0]
    verdict = 'SATISFIED' if pred == 1 else 'DISSATISFIED'
    print(f'{name:<15} -> Predicted: {verdict}')